In [0]:
%python
spark

# PART B – Bronze & Silver Layers

In [0]:
# Import important/required libraries
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
# 2. Start Spark
spark = SparkSession.builder.appName("SmartGear_Bronze_Silver_Gold").getOrCreate()

## Apply explicit schema (no inferSchema)

In [0]:
schema = StructType([
    StructField("OrderID", StringType(), False),
    StructField("OrderDate", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("StoreID", StringType(), False),
    StructField("Product", StringType(), False),
    StructField("Quantity", IntegerType(), False),
    StructField("UnitPrice", DoubleType(), False)
])

In [0]:
%python
# Define Medellion path structure

INPUT_PATH='/Volumes/workspace/default/smartgear_assignment_march2026/smartgear_sales.csv'

# Storage paths
BRONZE_PATH= "/Volumes/workspace/default/bronze/smartgear_assignment_march2026"
SILVER_PATH= "/Volumes/workspace/default/silver/smartgear_assignment_march2026"
GOLD_PATH= "/Volumes/workspace/default/gold/smartgear_assignment_march2026"

## Ingest raw CSV into Bronze layer using Databricks

In [0]:
bronze_raw = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv(INPUT_PATH) \

display(bronze_raw)

OrderID,OrderDate,Region,StoreID,Product,Quantity,UnitPrice
1001,2025-03-14,East,115,Headphones,3,81.65
1002,2025-02-19,West,110,Smartwatch,3,242.5
1003,2025-03-17,West,113,Printer,2,152.59
1004,2025-01-05,West,118,Camera,4,463.4
1005,2025-01-07,West,110,Tablet,3,370.01
1006,2025-03-22,East,113,Smartphone,4,639.47
1007,2025-02-19,North,118,Drone,4,747.21
1008,2025-02-21,West,112,Printer,2,151.06
1009,2025-03-30,North,117,Smartphone,1,614.89
1010,2025-01-30,West,108,Tablet,3,415.43


## Handle nulls and invalid records.

In [0]:
bronze_raw = bronze_raw.filter(
    (col("OrderID").isNotNull()) &
    (col("StoreID").isNotNull()) &
    (col("Product").isNotNull()) &
    (col("Quantity").isNotNull()) &
    (col("UnitPrice").isNotNull()) &
    (col("Quantity") > 0) &
    (col("UnitPrice") > 0))

## Parse OrderDate and create Revenue

In [0]:
bronze_df = bronze_raw \
    .withColumn("OrderDate", to_date(col("OrderDate"), "yyyy-MM-dd")) \
    .withColumn("Revenue", round(col("Quantity") * col("UnitPrice"), 2)) \
    .withColumn("year", year("OrderDate")) \
    .withColumn("month", month("OrderDate"))
display(bronze_df)

OrderID,OrderDate,Region,StoreID,Product,Quantity,UnitPrice,Revenue,year,month
1001,2025-03-14,East,115,Headphones,3,81.65,244.95,2025,3
1002,2025-02-19,West,110,Smartwatch,3,242.5,727.5,2025,2
1003,2025-03-17,West,113,Printer,2,152.59,305.18,2025,3
1004,2025-01-05,West,118,Camera,4,463.4,1853.6,2025,1
1005,2025-01-07,West,110,Tablet,3,370.01,1110.03,2025,1
1006,2025-03-22,East,113,Smartphone,4,639.47,2557.88,2025,3
1007,2025-02-19,North,118,Drone,4,747.21,2988.84,2025,2
1008,2025-02-21,West,112,Printer,2,151.06,302.12,2025,2
1009,2025-03-30,North,117,Smartphone,1,614.89,614.89,2025,3
1010,2025-01-30,West,108,Tablet,3,415.43,1246.29,2025,1


## Standardize Region

In [0]:
region_map = {"north": "North", "south": "South", "east": "East", "west": "West"}
bronze_df = bronze_df \
    .withColumn("Region", trim(lower(col("Region")))) \
    .replace(region_map, subset=["Region"])

display(bronze_df["Region"])

Column<'Region'>

## Add ingestion metadata

In [0]:
bronze_df = bronze_df \
    .withColumn("ingestion_timestamp", lit(datetime.now())) \
    .withColumn("ingestion_source", lit("Volumes/smartgear_sales.csv")) \
    .withColumn("ingestion_user", lit("databricks")) \
    .withColumn("ingestion_batch_id", lit("batch_202603"))

## Write Bronze layer (partitioned by year/month)

In [0]:
dbutils.fs.rm(BRONZE_PATH, recurse=True)

bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("year", "month") \
    .save(BRONZE_PATH)

print("✅ Bronze layer written to:", BRONZE_PATH)

✅ Bronze layer written to: /Volumes/workspace/default/bronze/smartgear_assignment_march2026


## Write cleaned data into Silver layer with partitioning

In [0]:
bronze_df.write \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .save(SILVER_PATH)

print("✅ Silver layer written to:", SILVER_PATH)

✅ Silver layer written to: /Volumes/workspace/default/silver/smartgear_assignment_march2026


# PART C – Gold Executive Layer

## Read cleaned Silver data(business‑ready transactions)

In [0]:
silver_df = spark.read \
    .load(SILVER_PATH)

silver_df = silver_df.select(
    "OrderID",
    "OrderDate",
    "Region",
    "StoreID",
    "Product",
    "Quantity",
    "UnitPrice",
    "Revenue",
    "year",
    "month"
)

silver_df.printSchema()
silver_df.show(5)

root
 |-- OrderID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- Region: string (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+-------+----------+------+-------+----------+--------+---------+-------+----+-----+
|OrderID| OrderDate|Region|StoreID|   Product|Quantity|UnitPrice|Revenue|year|month|
+-------+----------+------+-------+----------+--------+---------+-------+----+-----+
|   1001|2025-03-14|  East|    115|Headphones|       3|    81.65| 244.95|2025|    3|
|   1003|2025-03-17|  West|    113|   Printer|       2|   152.59| 305.18|2025|    3|
|   1006|2025-03-22|  East|    113|Smartphone|       4|   639.47|2557.88|2025|    3|
|   1009|2025-03-30| North|    117|Smartphone|       1|   614.89| 614.89|2025|    3|
|   1012|

## Create region‑level KPI dataset

In [0]:
region_kpi = silver_df \
    .groupBy("Region") \
    .agg(
        round(sum("Revenue"),2).alias("TotalRevenue"),
        sum("Quantity").alias("TotalQuantity"),
        round(avg("UnitPrice"),2).alias("AvgUnitPrice"),
        count("*").alias("TotalOrders"),
        countDistinct("StoreID").alias("StoresCount")
    ) \
    .withColumn("KPI_CreatedAt", lit(datetime.now())) \
    .withColumn("KPI_Level", lit("Region"))

## Write to Gold

In [0]:
region_kpi_path = f"{GOLD_PATH}/region_kpis"
region_kpi.write.mode("overwrite").save(region_kpi_path)

print("Region KPIs written to:", region_kpi_path)

display(region_kpi)

Region KPIs written to: /Volumes/workspace/default/gold/smartgear_assignment_march2026/region_kpis


Region,TotalRevenue,TotalQuantity,AvgUnitPrice,TotalOrders,StoresCount,KPI_CreatedAt,KPI_Level
North,353551.96,848,425.42,280,20,2026-03-24T05:55:16.299Z,Region
East,277434.47,719,387.22,240,20,2026-03-24T05:55:16.299Z,Region
South,313016.64,740,418.85,248,20,2026-03-24T05:55:16.299Z,Region
West,274303.2,694,392.51,232,20,2026-03-24T05:55:16.299Z,Region


## Generate Top 5 products by revenue

In [0]:
top_products = silver_df \
    .groupBy("Product") \
    .agg(
        round(sum("Revenue"),2).alias("ProductRevenue"),
        sum("Quantity").alias("TotalQuantity"),
        count("*").alias("TotalOrders")
    ) \
    .orderBy(desc("ProductRevenue")) \
    .limit(5)

top_products = top_products \
    .withColumn("Rank", row_number().over(Window.orderBy(desc("ProductRevenue")))) \
    .withColumn("KPI_CreatedAt", lit(datetime.now())) \
    .withColumn("KPI_Level", lit("Top5_Products"))

top_products_path = f"{GOLD_PATH}/top5_products_by_revenue"
top_products.write.mode("overwrite").save(top_products_path)

print("Top 5 products by revenue written to:", top_products_path)

display(top_products)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Top 5 products by revenue written to: /Volumes/workspace/default/gold/smartgear_assignment_march2026/top5_products_by_revenue


Product,ProductRevenue,TotalQuantity,TotalOrders,Rank,KPI_CreatedAt,KPI_Level
Laptop,207147.61,259,88,1,2026-03-24T05:55:19.312Z,Top5_Products
Smartphone,181489.55,301,99,2,2026-03-24T05:55:19.312Z,Top5_Products
Drone,179999.84,258,88,3,2026-03-24T05:55:19.312Z,Top5_Products
Camera,168943.84,337,111,4,2026-03-24T05:55:19.312Z,Top5_Products
Tablet,152735.88,379,120,5,2026-03-24T05:55:19.312Z,Top5_Products


## Gold tables (for reporting)
After writing the Parquet/Delta files, you can also expose them as SQL tables in Databricks

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.region_kpis (
  Region STRING,
  TotalRevenue DOUBLE,
  TotalQuantity BIGINT,
  AvgUnitPrice DOUBLE,
  TotalOrders BIGINT,
  StoresCount BIGINT,
  KPI_CreatedAt TIMESTAMP,
  KPI_Level STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.default.top5_products_by_revenue (
  Product STRING,
  ProductRevenue DOUBLE,
  TotalQuantity BIGINT,
  TotalOrders BIGINT,
  Rank INT,
  KPI_CreatedAt TIMESTAMP,
  KPI_Level STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.default.store_performance (
  StoreID STRING,
  TotalRevenue DOUBLE,
  TotalQuantity BIGINT,
  TotalOrders BIGINT
)
USING DELTA;

## Create product‑level and write it into Gold layer

In [0]:
# Product summary
product_summary = bronze_df \
    .groupBy("Product") \
    .agg(
        sum("Quantity").alias("TotalQuantity"),
        round(sum("Revenue"),2).alias("TotalRevenue"),
        round(avg("UnitPrice"),2).alias("AvgUnitPrice"),
        count("*").alias("TotalOrders")
    )
display(product_summary)


Product,TotalQuantity,TotalRevenue,AvgUnitPrice,TotalOrders
Printer,298,44713.98,150.16,106
Tablet,379,152735.88,403.56,120
Monitor,314,62501.48,199.0,104
Smartwatch,257,63432.84,246.94,85
Smartphone,301,181489.55,601.37,99
Headphones,306,24618.98,80.75,98
Laptop,259,207147.61,803.66,88
Camera,337,168943.84,500.87,111
Gaming Console,292,132722.27,452.74,101
Drone,258,179999.84,698.46,88


In [0]:
# 12. Write Gold layer (summarized tables)
product_summary.write.mode("overwrite").save(f"{GOLD_PATH}/product_summary")

print("✅ Gold layer written to:", GOLD_PATH)

✅ Gold layer written to: /Volumes/workspace/default/gold/smartgear_assignment_march2026


In [0]:
# for spark path='abfss://<Container>@<storage>.dfs.core.windows.net/<folder>/<file>'
# for sql path= https://<storage>.dfs.core.windows.net/<container>/<folder>/<file>
# https://smartgearassignment.dfs.core.windows.net/synapse-assignment/gold/smartgear_sales.csv
# storage account name: smartgearassignment
# container name:synapse-assignment 
# path = 'abfss://synapse-assignment@smartgearassignment.dfs.core.windows.net/raw/smartgear_sales.csv'

/Volumes/workspace/default/gold/smartgear_assignment_march2026/

# PART F – Data Quality & Engineering Reflection

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, sum as _sum

spark = SparkSession.getActiveSession()

# Input and storage paths
INPUT_PATH = '/Volumes/workspace/default/smartgear_assignment_march2026/smartgear_sales.csv'
BRONZE_PATH = "/Volumes/workspace/default/bronze/smartgear_assignment_march2026"
SILVER_PATH = "/Volumes/workspace/default/silver/smartgear_assignment_march2026"
GOLD_PATH = "/Volumes/workspace/default/gold/smartgear_assignment_march2026"

## 1. Read raw CSV

In [0]:
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(INPUT_PATH)

## 2. Data quality checks
###  a) Detect invalid Quantity or UnitPrice (<= 0, or NULL)

In [0]:
df_qc = df_raw.withColumn("invalid_quantity", 
    when(col("Quantity").isNotNull() & (col("Quantity") <= 0), True).otherwise(False)
).withColumn("invalid_unitprice",
    when(col("UnitPrice").isNotNull() & (col("UnitPrice") <= 0), True).otherwise(False)
)

### b) Count invalid / total records

In [0]:
q_summary = df_qc.agg(
    count("*").alias("total_records"),
    count(when(col("invalid_quantity") == True, True)).alias("invalid_quantity_count"),
    count(when(col("invalid_unitprice") == True, True)).alias("invalid_unitprice_count"),
    _sum(when(col("invalid_quantity") == True, 1).otherwise(0)).alias("invalid_quantity_total"),
    _sum(when(col("invalid_unitprice") == True, 1).otherwise(0)).alias("invalid_unitprice_total")
).collect()[0]

print("=== Data Quality Summary (SMARTGEAR) ===")
print(f"Total records: {q_summary['total_records']}")
print(f"Invalid (Quantity <= 0): {q_summary['invalid_quantity_count']}")
print(f"Invalid (UnitPrice <= 0): {q_summary['invalid_unitprice_count']}")
print("==========================================")

=== Data Quality Summary (SMARTGEAR) ===
Total records: 1000
Invalid (Quantity <= 0): 0
Invalid (UnitPrice <= 0): 0
